In [ ]:
import os
import json
import time
import re
import difflib
from textwrap import dedent
from dotenv import load_dotenv

load_dotenv()

from openai import OpenAI

DPO_DATASET_DIR = "./finetuning_data/crm-dpo-dataset/"
SFT_DATASET_DIR = "./finetuning_data/crm-sft-dataset/"

PERSONAS_PATH = "./data/personas.json"
os.makedirs(SFT_DATASET_DIR, exist_ok=True)

persona_tokens = []
if os.path.exists(PERSONAS_PATH):
    with open(PERSONAS_PATH, "r", encoding="utf-8") as f:
        personas = json.load(f)
    persona_tokens = sorted({p.get("name", "") for p in personas if p.get("name")})


# 기존 DPO 데이터셋 파일 읽기
with open(os.path.join(DPO_DATASET_DIR, "cycle_01_v2.json"), "r") as f:
    data = json.load(f)
    print(f"로드된 DPO 데이터 수 : {len(data)}")
    print(f"DPO 데이터 컬럼 들 : {list(data[0].keys())}")
    print(data[:10])
    print(type(data), type(f))

# SFT용 추가 프롬프트 정의 (tone_correction.py 기반 업데이트)
output_template = dedent("""
아래 출력 규칙과 컨텍스트를 바탕으로 CRM 메시지를 재작성하라.

## 필수 출력 형식 규칙
1) 제목은 한 줄 분량(20~25자)으로 간결하게 작성되어야 한다.
2) 본문은 두 줄~세 줄 분량(80~120자)으로 작성되어야 한다.
3) 본문의 마지막 한 줄은 반드시 CTA(Call-to-Action) 문장이어야 한다.
4) 출력 형식은 반드시 '[제목]'과 '[본문]' 레이블을 사용해야 한다.

## 금지 요소
- 영어 사용 금지: 브랜드/제품 고유명 제외, 한국어로만 작성
  (금지 예시: everyday, daily, routine, essential, ultimate, available 등)
- 페르소나/개인정보 직접 호명 금지
  (금지 예시: CL_01님, Budget_Seeker님, user_name 등)
- 메타 표현/특수문자 금지: { } ( ) * : ; # > < % _ 등
- JSON/코드 조각 금지: ```json, ### 등
- 과도한 이모지/구분선 금지

## 표현 수위 가이드
- 허용: '도움을 줄 수 있어요', '편안하게 느껴질 수 있어요', '부담 없이 사용하기 좋아요'
- 지양: '완벽 개선', '즉시 효과', '100% 보장', '치료' 등 의학적/단정 표현

[출력 형식]
[제목] 제목 내용
[본문] 본문 내용
""")

MODEL_NAME = "gpt-5-nano"
BATCH_SIZE = 1
MAX_OUTPUT_TOKENS = 1024
RETRY_LIMIT = 3

DEBUG = False
DEBUG_MAX_CHARS = 500

# 글자 수 기준 업데이트 (tone_correction.py 기반)
MIN_TITLE_LEN = 15  # 최소 제목 길이 (20-25자 권장)
MAX_TITLE_LEN = 30  # 최대 제목 길이
MIN_BODY_LEN = 60   # 최소 본문 길이 (80-120자 권장)
MAX_BODY_LEN = 150  # 최대 본문 길이

client = OpenAI()

def _debug(message):
    if DEBUG:
        print(message)

def _response_text(response):
    if hasattr(response, "output_text"):
        text = response.output_text
        if isinstance(text, list):
            return "\n".join(text)
        return text
    try:
        parts = []
        for item in response.output:
            if getattr(item, "type", None) == "message":
                for content in item.content:
                    if hasattr(content, "text"):
                        parts.append(content.text)
        return "\n".join(parts)
    except Exception:
        return ""

def rule_based_clean(text, persona_tokens):
    """규칙 기반 텍스트 정제 (tone_correction.py 금지 요소 기반)"""
    if not text:
        return text
    cleaned = text
    # JSON/코드 조각 제거
    cleaned = re.sub(r"```[a-zA-Z]*", "", cleaned)
    cleaned = cleaned.replace("```", "")
    cleaned = re.sub(r"(?m)^\s*(json|text)\s*$", "", cleaned)
    # 메타 표현/특수문자 제거
    cleaned = cleaned.translate(str.maketrans("", "", "{}[]"))
    cleaned = re.sub(r"\"?title\"?\s*:", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\"?body\"?\s*:", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\[\s*제목\s*\]|\[\s*본문\s*\]|CTA\s*:", "", cleaned)
    # 페르소나 직접 호명 제거
    if persona_tokens:
        pattern = r"(?<!\w)(?:" + "|".join(map(re.escape, persona_tokens)) + r")(?:님)?(?!\w)"
        cleaned = re.sub(pattern, "", cleaned)
    cleaned = re.sub(r"\S+님을\s*위해", "", cleaned)
    cleaned = re.sub(r"\S+분들께", "", cleaned)
    # 금지된 영어 표현 제거 (브랜드/제품명 제외)
    forbidden_english = [
        "everyday", "daily", "routine", "essential", "ultimate", 
        "available", "special", "premium", "luxury", "exclusive"
    ]
    for word in forbidden_english:
        cleaned = re.sub(rf"\b{word}\b", "", cleaned, flags=re.IGNORECASE)
    # 공백 정리
    cleaned = re.sub(r"[ \t]{2,}", " ", cleaned)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)
    return cleaned.strip()

def extract_removed_parts(original, cleaned, limit=5):
    matcher = difflib.SequenceMatcher(None, original, cleaned)
    parts = []
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag in ("delete", "replace"):
            removed = original[i1:i2]
            removed = re.sub(r"\s+", " ", removed).strip()
            if removed:
                if len(removed) > 80:
                    removed = removed[:77] + "..."
                parts.append(removed)
    seen = set()
    uniq = []
    for part in parts:
        if part not in seen:
            seen.add(part)
            uniq.append(part)
    return uniq[:limit]

def _len_in_range(text, min_len, max_len):
    """글자 수가 범위 내에 있는지 확인"""
    if text is None:
        return False
    compact = re.sub(r"\s+", "", text)
    return min_len <= len(compact) <= max_len

def _min_len_ok(text, min_len):
    if text is None:
        return False
    compact = re.sub(r"\s+", "", text)
    return len(compact) >= min_len

def normalize_refined(text):
    """GPT 출력을 정규화하여 [제목]/[본문] 형식으로 변환"""
    if not text:
        return ""
    cleaned = text.strip()
    cleaned = re.sub(r"```[a-zA-Z]*", "", cleaned)
    cleaned = cleaned.replace("```", "").strip()
    title = None
    body = None
    # Try JSON extraction first
    if "{" in cleaned and "}" in cleaned:
        start = cleaned.find("{")
        end = cleaned.rfind("}") + 1
        try:
            obj = json.loads(cleaned[start:end])
            if isinstance(obj, dict):
                title = str(obj.get("title", "")).strip() or None
                body = str(obj.get("body", "")).strip() or None
        except Exception:
            pass
    # Fallback to label lines
    if title is None or body is None:
        lines = [line.strip() for line in cleaned.splitlines() if line.strip()]
        for line in lines:
            # [제목] 또는 제목: 형식 파싱
            if line.startswith("[제목]"):
                title = line.split("[제목]", 1)[1].strip() or title
            elif line.startswith("제목:"):
                title = line.split("제목:", 1)[1].strip() or title
            elif line.lower().startswith("title:"):
                title = line.split(":", 1)[1].strip() or title
            # [본문] 또는 본문: 형식 파싱
            elif line.startswith("[본문]"):
                body = line.split("[본문]", 1)[1].strip() or body
            elif line.startswith("본문:"):
                body = line.split("본문:", 1)[1].strip() or body
            elif line.lower().startswith("body:"):
                body = line.split(":", 1)[1].strip() or body
        if title is None and lines:
            title = lines[0]
        if body is None and len(lines) > 1:
            body = " ".join(lines[1:])
    # 길이 검증 (tone_correction.py 기준)
    if not _min_len_ok(title, MIN_TITLE_LEN) or not _min_len_ok(body, MIN_BODY_LEN):
        return ""
    result = []
    result.append(f"[제목] {title}")
    result.append(f"[본문] {body}")
    return "\n".join(result).strip()

def refine_message(context, draft, output_rules):
    system = output_rules.strip()
    user = f"{context}\n\n{draft}\n"
    last_error = None
    for attempt in range(RETRY_LIMIT):
        try:
            _debug(f"[GPT] attempt={attempt + 1}/{RETRY_LIMIT} model={MODEL_NAME}")
            _debug(f"[GPT] system_len={len(system)} user_len={len(user)}")
            _debug(f"[GPT] system_preview=\n{system[:DEBUG_MAX_CHARS]}")
            _debug(f"[GPT] user_preview=\n{user[:DEBUG_MAX_CHARS]}")
            response = client.responses.create(
                reasoning={"effort": "low"},
                model=MODEL_NAME,
                input=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
                max_output_tokens=MAX_OUTPUT_TOKENS,
            )
            _debug(f"[GPT] response_status={getattr(response, 'status', None)} incomplete={getattr(response, 'incomplete_details', None)}")
            text = _response_text(response).strip()
            text = normalize_refined(text)
            _debug(f"[GPT] output_text_len={len(text)}")
            _debug(f"[GPT] output_text_preview=\n{text[:DEBUG_MAX_CHARS]}")
            if DEBUG:
                try:
                    dump = response.model_dump()
                    _debug(f"[GPT] response_dump_preview=\n{json.dumps(dump, ensure_ascii=False)[:DEBUG_MAX_CHARS]}")
                except Exception:
                    _debug(f"[GPT] response_repr_preview=\n{str(response)[:DEBUG_MAX_CHARS]}")
            if text:
                return text
        except Exception as exc:
            last_error = exc
            _debug(f"[GPT] error={type(exc).__name__}: {exc}")
            time.sleep(2 ** attempt)
    if last_error:
        raise last_error
    return ""

In [ ]:
# 데이터 갯수 확인 이후 남은 데이터 refine
with open(os.path.join(SFT_DATASET_DIR, "cycle_01_v2.jsonl"), "r") as f:
    remain_data_num = sum(1 for _ in f)
    print(f"저장된 SFT 데이터 수 : {remain_data_num}")

# SFT 프롬프트용 출력 규칙 (tone_correction.py 기반 업데이트)
output_template_for_sft = dedent("""
## 필수 출력 형식 규칙
1) 제목은 한 줄 분량(20~25자)으로 간결하게 작성
2) 본문은 두 줄~세 줄 분량(80~120자)으로 작성
3) 본문 마지막은 CTA(Call-to-Action) 문장 필수
4) 출력 형식: [제목], [본문] 레이블 사용

## 금지 요소
- 영어 사용 금지 (브랜드/제품 고유명 제외)
- 페르소나/개인정보 직접 호명 금지
- 메타 표현/특수문자/이모지 금지

[출력 형식]
[제목] 제목 내용
[본문] 본문 내용
""")

total_batches = (len(data[remain_data_num:]) + BATCH_SIZE - 1) // BATCH_SIZE
for batch_index in range(total_batches):
    refined_data = []
    start = batch_index * BATCH_SIZE + remain_data_num
    batch = data[start : start + BATCH_SIZE]
    print(f"Batch {batch_index + 1}/{total_batches}")

    for idx_in_batch, example in enumerate(batch):
        sample_index = start + idx_in_batch + 1
        context = example.get("prompt", "")
        draft = example.get("chosen", "")
        cleaned = rule_based_clean(draft, persona_tokens)
        if cleaned != draft:
            removed_parts = extract_removed_parts(draft, cleaned)
            if removed_parts:
                print(f"[정제됨] {sample_index}번: " + ", ".join(removed_parts))
            else:
                print(f"[정제됨] {sample_index}번: 변경 감지(공백/형식)")
        refined = refine_message(context, cleaned, output_template)
        if not refined:
            print(f"[제외] {sample_index}번: 제목/본문 길이 부족")
            continue
        refined_data.append({**example, "chosen_refined": refined})

    with open(os.path.join(SFT_DATASET_DIR, "cycle_01_v2.jsonl"), "a") as f:
        for example in refined_data:
            prompt = example.get("prompt", "")
            chosen = example.get("chosen_refined", example.get("chosen", ""))
            prompt = "다음 조건에 맞는 CRM 메시지를 작성하세요. " + prompt + f" {output_template_for_sft.replace(chr(10), ' ')}"
            f.write(json.dumps({"prompt": prompt, "chosen": chosen}, ensure_ascii=False) + "\n")